# 📘 Logistic Regression — Concepts Notebook
### Atlas Skilltech University | Instructor Reference

---

**Yeh notebook kyun hai?** Yahan hum Logistic Regression ke har topic ko chhote chhote sample data ke saath samjhenge — taaki concepts ekdum clear ho jayein real dataset pe jaane se pehle.

**Structure (Kya kya cover karenge):**
1. Logistic Regression kya hota hai?
2. Logistic (Sigmoid) Function
3. Linear se Logistic — intuition aur math
4. Maximum Likelihood se best fit
5. EDA aur Model Training
6. Confusion Matrix aur Accuracy
7. Precision, Recall, F1 Score
8. ROC Curves
9. Performance Evaluation Summary
10. Multiclass Classification

> 💡 Har mushkil keyword ko code se **pehle** markdown cell mein explain kiya gaya hai.

---
## 🔧 Step 0 — Libraries Import Karo

**Yeh keywords yaad rakho:**
- **numpy** — numbers aur math operations ke liye library
- **pandas** — tables (DataFrames) ke liye library
- **matplotlib** — charts banane ke liye library
- **scikit-learn (sklearn)** — Python ka sabse popular machine learning library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Saari libraries successfully import ho gayi ✅')

---
## 📌 Section 1 — Logistic Regression Kya Hota Hai?

**Yeh keywords yaad rakho:**
- **Classification** — ek aisa machine learning task jahan hum koi *category* predict karte hain (jaise Pass/Fail, Yes/No, Spam/Not Spam)
- **Binary Classification** — jab sirf **2 categories** hoti hain (0 ya 1)
- **Dependent variable (target)** — woh column jo hum predict karna chahte hain (jaise "Pass")
- **Independent variables (features)** — woh columns jo hum prediction karne ke liye use karte hain (jaise "Hours Studied")

Naam mein "Regression" hai, lekin **Logistic Regression Classification ke liye use hota hai**, regression ke liye nahi.
Yeh predict karta hai ki koi cheez kisi category mein aane ki **probability** kitni hai.

**Example:** Ek student ne jitne ghante padha, uski basis pe woh exam pass karega ya nahi — yahi predict karna hai.

In [ ]:
# Sample dataset — Student Exam Pass/Fail
data = {
    'Hours_Studied': [1, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8, 9],
    'Pass':          [0, 0, 0,   0, 0,   0, 1,   1, 1,   1, 1,   1, 1,   1, 1]
}

df = pd.DataFrame(data)
print(df)
print('\nData ka shape:', df.shape)
print('\nTarget distribution (0=Fail, 1=Pass):')
print(df['Pass'].value_counts())

In [ ]:
# Data ko visualise karte hain
plt.figure(figsize=(8, 4))
plt.scatter(df['Hours_Studied'], df['Pass'], color='steelblue', s=80, zorder=5)
plt.xlabel('Kitne Ghante Padha')
plt.ylabel('Pass (1) / Fail (0)')
plt.title('Student ka Exam Result vs Padhai ke Ghante')
plt.yticks([0, 1], ['Fail (0)', 'Pass (1)'])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 📌 Section 2 — Logistic (Sigmoid) Function

**Yeh keywords yaad rakho:**
- **Sigmoid function** — ek S-shape wala curve jo kisi bhi number ko (0, 1) ke beech mein squeeze kar deta hai. Probability ke liye perfect!
- **e** — Euler's number ≈ 2.718 (ek mathematical constant)
- **z** — raw score (koi bhi number, -∞ se +∞ tak ho sakta hai)

**Formula:**
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

- Jab z → +∞ hoga, σ(z) → 1 ho jaayega
- Jab z → -∞ hoga, σ(z) → 0 ho jaayega
- Jab z = 0 hoga, σ(z) = 0.5 hoga

Matlab sigmoid function koi bhi raw score ko **0 aur 1 ke beech ki probability** mein convert kar deta hai.

In [ ]:
# Sigmoid function define karte hain
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Kuch values test karte hain
test_values = [-10, -3, 0, 3, 10]
print('z ki value  |  sigmoid(z)')
print('-' * 25)
for z in test_values:
    print(f'  {z:>5}       |  {sigmoid(z):.4f}')

In [ ]:
# Sigmoid curve plot karte hain
z = np.linspace(-10, 10, 300)

plt.figure(figsize=(8, 4))
plt.plot(z, sigmoid(z), color='darkorange', linewidth=2.5, label='Sigmoid σ(z)')
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.6, label='Threshold = 0.5')
plt.axvline(0, color='gray', linestyle='--', alpha=0.3)
plt.xlabel('z  (raw score)')
plt.ylabel('σ(z)  (probability)')
plt.title('Sigmoid / Logistic Function ka Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 📌 Section 3 — Linear Regression se Logistic Regression tak

### Linear Regression classification ke liye kyun kaam nahi karta?

**Yeh keywords yaad rakho:**
- **Linear Regression** — continuous number predict karta hai (jaise ghar ki kimat)
- **Threshold** — ek cutoff point; agar probability > 0.5 ho toh class 1 predict karo, warna class 0
- **Log-Odds / Logit** — odds ratio ka natural log

**Linear Regression ka equation:**
$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \ldots$$

Problem: $\hat{y}$ negative bhi ho sakta hai ya 1 se bada bhi — probability ke liye yeh sahi nahi!

**Logistic Regression ka fix:** Hum linear equation ko sigmoid ke andar daal dete hain:
$$P(y=1) = \sigma(w_0 + w_1 x_1) = \frac{1}{1 + e^{-(w_0 + w_1 x_1)}}$$

**Log-Odds (Logit) form:**
$$\ln\left(\frac{P}{1-P}\right) = w_0 + w_1 x_1$$

Matlab logistic regression actually **log-odds scale pe ek linear model** hai.

In [ ]:
from sklearn.linear_model import LinearRegression, LogisticRegression

X = df[['Hours_Studied']]
y = df['Pass']

# Linear regression fit karte hain
lin_reg = LinearRegression()
lin_reg.fit(X, y)

# Logistic regression fit karte hain
log_reg = LogisticRegression()
log_reg.fit(X, y)

x_range = np.linspace(0, 10, 300).reshape(-1, 1)

plt.figure(figsize=(10, 5))

# Left: Linear Regression
plt.subplot(1, 2, 1)
plt.scatter(df['Hours_Studied'], df['Pass'], color='steelblue', s=80, zorder=5)
plt.plot(x_range, lin_reg.predict(x_range), color='red', linewidth=2)
plt.axhline(0, color='gray', linestyle='--', alpha=0.5)
plt.axhline(1, color='gray', linestyle='--', alpha=0.5)
plt.title('❌ Linear Regression (classification ke liye bura)')
plt.xlabel('Padhai ke Ghante')
plt.ylabel('Pass Probability')
plt.ylim(-0.3, 1.3)
plt.grid(True, alpha=0.3)

# Right: Logistic Regression
plt.subplot(1, 2, 2)
plt.scatter(df['Hours_Studied'], df['Pass'], color='steelblue', s=80, zorder=5)
plt.plot(x_range, log_reg.predict_proba(x_range)[:, 1], color='darkorange', linewidth=2)
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Threshold 0.5')
plt.title('✅ Logistic Regression (classification ke liye sahi)')
plt.xlabel('Padhai ke Ghante')
plt.ylabel('Pass hone ki Probability')
plt.ylim(-0.1, 1.1)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 📌 Section 4 — Maximum Likelihood Estimation (MLE)

**Yeh keywords yaad rakho:**
- **Parameters (weights)** — woh numbers w₀, w₁, w₂, ... jo model sikhta hai
- **Likelihood** — woh probability ki model ne jo data observe kiya woh actually generate karta
- **Maximum Likelihood Estimation (MLE)** — woh parameter values dhundho jo observed data ko **sabse zyada probable** banayein
- **Log-Likelihood** — likelihood ka log lete hain (math aasaan ho jaati hai, optimal point same rehta hai)
- **Loss function (Log Loss / Binary Cross-Entropy)** — woh quantity jo hum minimize karte hain; yeh negative log-likelihood hai

**Log Loss formula:**
$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\hat{p}_i) + (1 - y_i) \log(1 - \hat{p}_i) \right]$$

Jahan:
- $y_i$ = actual label (0 ya 1)
- $\hat{p}_i$ = predicted probability
- $N$ = total samples

Gradient descent (ya similar optimization) weights ko **minimize** karne ke liye adjust karta hai.

In [ ]:
from sklearn.metrics import log_loss

# Probabilities predict karte hain
X = df[['Hours_Studied']]
y = df['Pass']

log_reg = LogisticRegression()
log_reg.fit(X, y)

probs = log_reg.predict_proba(X)[:, 1]

print('Sikhe hue weights (coefficients):', log_reg.coef_)
print('Intercept (bias / w0):', log_reg.intercept_)
print()
print('Log Loss (jitna kam utna better):', round(log_loss(y, probs), 4))

# Predicted probabilities aur actual labels ek saath dikhate hain
result_df = df.copy()
result_df['Predicted_Prob'] = probs.round(3)
result_df['Predicted_Class'] = (probs >= 0.5).astype(int)
print()
print(result_df)

---
## 📌 Section 5 — EDA aur Model Training (Proper Pipeline)

**Yeh keywords yaad rakho:**
- **EDA (Exploratory Data Analysis)** — modeling se pehle charts aur statistics se data explore karna
- **Train-Test Split** — data ko do hisson mein baantna: ek hisse pe model train karo, dusre pe performance check karo
- **Overfitting** — model training data yaad kar leta hai, naye data pe fail ho jaata hai
- **random_state** — ek seed number; ensure karta hai ki split har baar same ho
- **test_size=0.2** — 20% data testing ke liye, 80% training ke liye

Yahan hum thoda bada sample dataset use karenge.

In [ ]:
# Bada sample dataset — 100 students ke Hours Studied aur Previous Score
np.random.seed(42)
n = 100

hours    = np.round(np.random.uniform(1, 10, n), 1)
prev     = np.round(np.random.uniform(30, 95, n), 1)
# Pass agar hours > 5 YA prev > 65, thoda noise ke saath
prob_raw = 0.5 * (hours / 10) + 0.5 * (prev / 100)
noise    = np.random.normal(0, 0.1, n)
y_true   = ((prob_raw + noise) > 0.5).astype(int)

students_df = pd.DataFrame({
    'Hours_Studied': hours,
    'Previous_Score': prev,
    'Pass': y_true
})

print('Dataset ka shape:', students_df.shape)
print()
students_df.head(10)

In [ ]:
# --- EDA ---
print('Basic Statistics:')
print(students_df.describe())
print()
print('Missing values (khaali cells):')
print(students_df.isnull().sum())
print()
print('Class balance (Pass column):')
print(students_df['Pass'].value_counts())

In [ ]:
# EDA Charts
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Class distribution
students_df['Pass'].value_counts().plot(kind='bar', ax=axes[0], color=['salmon', 'steelblue'], edgecolor='black')
axes[0].set_title('Class Distribution (Kitne Pass, Kitne Fail)')
axes[0].set_xlabel('Pass (1) / Fail (0)')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Fail (0)', 'Pass (1)'], rotation=0)

# 2. Hours studied by outcome
pass_students = students_df[students_df['Pass'] == 1]['Hours_Studied']
fail_students = students_df[students_df['Pass'] == 0]['Hours_Studied']
axes[1].hist(pass_students, alpha=0.7, color='steelblue', label='Pass', bins=10)
axes[1].hist(fail_students, alpha=0.7, color='salmon', label='Fail', bins=10)
axes[1].set_title('Padhai ke Ghante — Pass vs Fail')
axes[1].set_xlabel('Padhai ke Ghante')
axes[1].legend()

# 3. Scatter
colors = ['salmon' if p == 0 else 'steelblue' for p in students_df['Pass']]
axes[2].scatter(students_df['Hours_Studied'], students_df['Previous_Score'], c=colors, s=50, alpha=0.7)
axes[2].set_title('Hours vs Previous Score (colour = result)')
axes[2].set_xlabel('Padhai ke Ghante')
axes[2].set_ylabel('Pehle ka Score')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

# Features (X) aur target (y)
X = students_df[['Hours_Studied', 'Previous_Score']]
y = students_df['Pass']

# Train-test split karte hain
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples  : {len(X_train)}')
print(f'Testing  samples  : {len(X_test)}')

In [ ]:
from sklearn.preprocessing import StandardScaler

# Scaling — keywords:
#   StandardScaler: har feature ko transform karta hai taaki mean=0, std=1 ho
#   Sirf training data pe fit karo, phir dono train aur test pe transform karo

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Model train karte hain
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

print('Model train ho gaya ✅')
print('Coefficients:', model.coef_)
print('Intercept   :', model.intercept_)

---
## 📌 Section 6 — Confusion Matrix aur Accuracy

**Yeh keywords yaad rakho:**
- **Confusion Matrix** — ek 2×2 table jo actual vs predicted values compare karta hai
- **True Positive (TP)** — model ne Pass bola, actually Pass tha ✅
- **True Negative (TN)** — model ne Fail bola, actually Fail tha ✅
- **False Positive (FP)** — model ne Pass bola, actually Fail tha ❌ (Type I Error)
- **False Negative (FN)** — model ne Fail bola, actually Pass tha ❌ (Type II Error)
- **Accuracy** — kitne predictions sahi the = (TP + TN) / Total

```
            Predicted Fail   Predicted Pass
Actual Fail      TN                FP
Actual Pass      FN                TP
```

In [ ]:
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score
)

y_pred = model.predict(X_test_scaled)

# Accuracy
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.2%}')

# Confusion matrix — numbers
cm = confusion_matrix(y_test, y_pred)
print('\nConfusion Matrix (raw numbers):')
print(cm)

In [ ]:
# --- TARIKA 1 (purana style): matplotlib heatmap ---
import seaborn as sns

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Fail', 'Pass'],
            yticklabels=['Fail', 'Pass'])
plt.xlabel('Predicted (Model ne kya bola)')
plt.ylabel('Actual (Actually kya tha)')
plt.title('Confusion Matrix — seaborn heatmap (purana tarika)')
plt.tight_layout()
plt.show()

In [ ]:
# --- TARIKA 2 (modern, recommended): ConfusionMatrixDisplay ---
# ConfusionMatrixDisplay sklearn mein built-in hai aur sab kuch automatically handle karta hai

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Fail', 'Pass'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — ConfusionMatrixDisplay (naya aur recommended tarika)')
plt.tight_layout()
plt.show()

---
## 📌 Section 7 — Precision, Recall, F1 Score

### Sirf Accuracy kyun kaafi nahi hoti?
Agar 95 students mein se 95 Pass karte hain, toh jo model hamesha "Pass" bolta hai woh 95% accuracy pata hai — lekin yeh toh bekar hai!
Hume better metrics chahiye.

**Yeh keywords yaad rakho:**
- **Precision** — model ne jitne Pass predict kiye, unme se actually kitne Pass the?
  $$\text{Precision} = \frac{TP}{TP + FP}$$
- **Recall (Sensitivity)** — jo actually Pass the, unme se model ne kitno ko sahi pakda?
  $$\text{Recall} = \frac{TP}{TP + FN}$$
- **F1 Score** — Precision aur Recall ka harmonic mean (dono balance karta hai):
  $$F_1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$
- **Classification Report** — ek table jo saare metrics ek saath dikhata hai
- **Support** — har class mein actual samples ki count

**Rule of thumb:**
- **Precision** tab use karo jab False Positives costly hoon (jaise spam filter)
- **Recall** tab use karo jab False Negatives costly hoon (jaise cancer diagnosis)
- **F1** tab use karo jab dono ka balance chahiye

In [ ]:
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

# Individual scores
print('Precision :', round(precision_score(y_test, y_pred), 4))
print('Recall    :', round(recall_score(y_test, y_pred), 4))
print('F1 Score  :', round(f1_score(y_test, y_pred), 4))
print()

# Full report
print('Pura Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Fail', 'Pass']))

---
## 📌 Section 8 — ROC Curve aur AUC

**Yeh keywords yaad rakho:**
- **Threshold** — woh cutoff probability jiske upar hum class 1 predict karte hain (default 0.5 hai)
- **True Positive Rate (TPR / Sensitivity / Recall)** — actual positives mein se kitne correctly identify hue
  $$TPR = \frac{TP}{TP + FN}$$
- **False Positive Rate (FPR)** — actual negatives mein se kitne galat positives bole
  $$FPR = \frac{FP}{FP + TN}$$
- **ROC Curve** — har possible threshold pe TPR vs FPR plot karta hai
- **AUC (Area Under the Curve)** — 0 se 1 ke beech ek single number; 1 ke paas ho toh better
  - AUC = 0.5 → random guessing
  - AUC = 1.0 → perfect model
- **RocCurveDisplay** — ROC curve plot karne ka modern sklearn tarika (recommended)

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, RocCurveDisplay

y_proba = model.predict_proba(X_test_scaled)[:, 1]

auc = roc_auc_score(y_test, y_proba)
print(f'AUC Score: {auc:.4f}')

In [ ]:
# --- TARIKA 1 (purana): roc_curve use karke manually plot karo ---
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='darkorange', linewidth=2, label=f'ROC Curve (AUC = {auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve — Manual (purana tarika)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- TARIKA 2 (modern, recommended): RocCurveDisplay ---
disp = RocCurveDisplay.from_estimator(model, X_test_scaled, y_test)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.title('ROC Curve — RocCurveDisplay (naya aur recommended tarika)')
plt.legend()
plt.tight_layout()
plt.show()

---
## 📌 Section 9 — Logistic Regression Performance Evaluation Summary

Yahan hum saare evaluation metrics ek clean summary mein daalenge.

**Yeh keywords yaad rakho:**
- **Specificity (True Negative Rate)** — TN / (TN + FP); model kitna accha negatives identify karta hai
- **Balanced Accuracy** — sensitivity aur specificity ka average; imbalanced datasets ke liye better

In [ ]:
from sklearn.metrics import balanced_accuracy_score

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

print('=' * 50)
print('     LOGISTIC REGRESSION — PERFORMANCE SUMMARY')
print('=' * 50)
print(f'  True Positives  (TP) : {tp}')
print(f'  True Negatives  (TN) : {tn}')
print(f'  False Positives (FP) : {fp}')
print(f'  False Negatives (FN) : {fn}')
print('-' * 50)
print(f'  Accuracy         : {accuracy_score(y_test, y_pred):.4f}')
print(f'  Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred):.4f}')
print(f'  Precision        : {precision_score(y_test, y_pred):.4f}')
print(f'  Recall           : {recall_score(y_test, y_pred):.4f}')
print(f'  F1 Score         : {f1_score(y_test, y_pred):.4f}')
print(f'  AUC-ROC          : {roc_auc_score(y_test, y_proba):.4f}')
print('=' * 50)

---
## 📌 Section 10 — Multiclass Classification with Logistic Regression

**Yeh keywords yaad rakho:**
- **Multiclass Classification** — 3 ya zyada categories mein se ek predict karna (jaise Grade A/B/C)
- **One-vs-Rest (OvR)** — har class ke liye ek binary classifier train karo (yeh class vs baaki sab)
- **Softmax** — sigmoid ka generalisation multiple classes ke liye; har class ke liye probability output karta hai
- **multi_class='multinomial'** — softmax use karta hai true multi-class model ke liye
- **macro avg** — har class ka metric equally average karta hai
- **weighted avg** — metric ko support (class size) se weight karke average karta hai

In [ ]:
# 3-class sample dataset banate hain — Grade A, B, C
np.random.seed(0)
n = 150

hours2  = np.round(np.random.uniform(1, 10, n), 1)
score2  = np.round(np.random.uniform(30, 100, n), 1)

# Combined score ke basis pe grade assign karte hain
combined = 0.4 * hours2 + 0.006 * score2
grade = np.where(combined >= 0.75, 2,   # Grade A
        np.where(combined >= 0.5,  1,   # Grade B
                                   0))  # Grade C

multi_df = pd.DataFrame({
    'Hours_Studied': hours2,
    'Previous_Score': score2,
    'Grade': grade
})

print('Class distribution:')
print(multi_df['Grade'].value_counts().sort_index())
print(multi_df.head())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay

X_m = multi_df[['Hours_Studied', 'Previous_Score']]
y_m = multi_df['Grade']

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_m, y_m, test_size=0.2, random_state=42
)

scaler_m = StandardScaler()
X_train_m_sc = scaler_m.fit_transform(X_train_m)
X_test_m_sc  = scaler_m.transform(X_test_m)

# Multi-class logistic regression
multi_model = LogisticRegression(multi_class='multinomial', solver='lbfgs', random_state=42)
multi_model.fit(X_train_m_sc, y_train_m)

y_pred_m = multi_model.predict(X_test_m_sc)

print('Multiclass Classification Report:')
print(classification_report(y_test_m, y_pred_m, target_names=['Grade C', 'Grade B', 'Grade A']))

In [ ]:
# Multiclass ke liye confusion matrix
ConfusionMatrixDisplay.from_estimator(
    multi_model, X_test_m_sc, y_test_m,
    display_labels=['Grade C', 'Grade B', 'Grade A'],
    cmap='Blues'
)
plt.title('Multiclass Confusion Matrix')
plt.tight_layout()
plt.show()

---
## ✅ Summary — Kya Kya Sikha

| Topic | Key Takeaway |
|---|---|
| Logistic Regression | Classification algorithm hai, Regression nahi |
| Sigmoid Function | Output ko (0,1) mein squeeze karta hai probability ke liye |
| Linear → Logistic | Log-odds scale pe linear model hota hai |
| MLE / Log Loss | Best weights dhundhne ka optimization method |
| Train-Test Split | Unseen data pe evaluate karo |
| Confusion Matrix | TP, TN, FP, FN ka breakdown |
| Precision / Recall / F1 | Accuracy se better metrics |
| ROC / AUC | Model ki overall discriminative power |
| Multiclass | 3+ classes ke liye One-vs-Rest ya Softmax |

---
*Aage: Yeh sab real Kaggle dataset pe apply karo → Notebook 2 dekho*